# Nairobi Flood–Waste–Displacement (30 m) — Clean Workflow

This notebook produces:
- **30 m population-change rasters** per event via dasymetric allocation (using your `nairobi_built_weight_30m`),
- **Flood exposure** metrics per event,
- A **2 km GeoDataFrame** (`gdf_map`) with per‑event stats and **waste** covariates,
- Optional pixel‑level analysis tables.

> Tip: run cells in order. Update only the **paths/config** in the first section.

## 0) Setup & Configuration

In [1]:
# !pip install geopandas rasterio shapely pyproj rasterstats tqdm scipy scikit-learn --quiet

import os, numpy as np, pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio import features
from rasterio.enums import Resampling
from rasterio.warp import reproject
from shapely.geometry import box, Point
from tqdm import tqdm
from scipy import ndimage
from pathlib import Path

WASTE_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

In [2]:
# ---- UPDATE THESE PATHS ----
FLOOD_RASTER = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/FastFlood/model8.tif"          # 30 m flood raster (binary or depth)
BUILT_WEIGHT = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Clean/building_weight_30m_area_only.tif"  # from your GEE export, aligned to flood if possible
POP_GRID_2KM = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Clean/FB_n_3857.gpkg"     # vector polygons with a pop count field
# WASTE_POINTS = "data/waste_points.geojson"               # waste points (optional; can skip section)

# Event fields (displacement/change) in the 2 km layer:
POP_FIELDS = ['density_diff_043016_050716', 'n_diff_043016_050716', 
              'density_diff_050100_050800', 'n_diff_050100_050800', 
              'density_diff_050208_050908', 'n_diff_050208_050908', 
              'variation_density_sum', 'variation_density_abs_sum', 
              'variation_count_sum', 'variation_count_abs_sum',] 

# Name of the *single* numeric field if you want to run only one event quickly:
# POP_FIELDS = ["pop_change_total"]

# Flood handling:
FLOOD_IS_BINARY = False          # set True if flood raster is already 0/1
FLOOD_DEPTH_THRESHOLD = 0.10     # meters; used only when FLOOD_IS_BINARY=False

# Waste KDE bandwidth (meters):
WASTE_KDE_BW_M = 500.0           # try 250 / 500 / 1000 in sensitivity

# Outputs
os.makedirs(WASTE_DIR/"outputs/rasters_30m", exist_ok=True)
os.makedirs(WASTE_DIR/"outputs/waste", exist_ok=True)
OUTPUT_STATS_2KM = WASTE_DIR/"outputs/zonal_stats_2km.csv"

In [3]:
# Waste location
waste_df = pd.read_csv("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/Correct_SVI.csv")
waste_df = waste_df[(waste_df['img_dir'] != 'ZWL/') & (waste_df['img_dir'] != 'Faith/')]
# waste_df

# Convert to GeoDataFrame
waste_gdf = gpd.GeoDataFrame(
    waste_df,
    geometry=gpd.points_from_xy(waste_df["lon"], waste_df["lat"]),
    crs="EPSG:4326"  # WGS84 (lat/lon). Change if your coords are in another CRS
)
waste_gdf = waste_gdf.to_crs('EPSG:3857')  # align to 30 m grid CRS
# waste_gdf = waste_gdf.to_crs(flood_crs)  # align to 30 m grid CRS
waste_gdf

,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,yolo_conf,yolo_bbox,yolo_num,prediction,Domestic,Construction,geometry
73,1Xyfr0j4E3Gx6_Coadexow_180,2018,2,NaN,NaN,-1.282457,36.751317,1Xyfr0j4E3Gx6_Coadexow,Google/1/X/,True,[0.4008658230304718],"[[298.3153076171875, 184.95751953125, 400.0, 2...",1,yes,Y,Y,POINT (4091137.948 -142774.33)
74,nTRlyH7aG8CmTfOI1_akgw_0,2018,3,NaN,NaN,-1.279155,36.719541,nTRlyH7aG8CmTfOI1_akgw,Google/n/T/,True,"[0.5433366894721985, 0.3535045087337494]","[[50.18629455566406, 221.9884033203125, 188.91...",2,yes,Y,N,POINT (4087600.597 -142406.681)
75,uc2yRcVOf6bCSyBKfnm0Sw_90,2018,2,NaN,NaN,-1.283062,36.751316,uc2yRcVOf6bCSyBKfnm0Sw,Google/u/c/,True,"[0.3149861693382263, 0.2503598630428314]","[[0.0, 248.11912536621094, 191.11526489257812,...",2,yes,Y,Y,POINT (4091137.747 -142841.789)
76,y_-_BCz3RPfZPFkLoqlA7Q_0,2018,2,NaN,NaN,-1.285056,36.745694,y_-_BCz3RPfZPFkLoqlA7Q,Google/y/_/,True,[0.3682653307914734],"[[185.56979370117188, 191.65072631835938, 309....",1,yes,Y,Y,POINT (4090511.941 -143063.76)
77,y_-_BCz3RPfZPFkLoqlA7Q_90,2018,2,NaN,NaN,-1.285056,36.745694,y_-_BCz3RPfZPFkLoqlA7Q,Google/y/_/,True,[0.27535533905029297],"[[71.40460968017578, 221.17556762695312, 324.6...",1,yes,Y,Y,POINT (4090511.941 -143063.76)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3965,K2bDOcQT-fC73J3sTszcJw_0,2021,8,NaN,NaN,-1.314196,36.889760,K2bDOcQT-fC73J3sTszcJw,Google/K/2/,True,[0.5377181172370911],"[[322.23724365234375, 0.0, 399.8905944824219, ...",1,yes,Y,Y,POINT (4106549.279 -146308.441)
3966,A2MMbVUET-qhK8hNcteVBw_180,2018,2,NaN,NaN,-1.276726,36.919951,A2MMbVUET-qhK8hNcteVBw,Google/A/2/,True,[0.8420240879058838],"[[2.8917694091796875, 153.89686584472656, 383....",1,yes,Y,N,POINT (4109910.15 -142136.295)
3967,zVDYat_guuNnsZ9DemVKwA_180,2018,2,NaN,NaN,-1.276957,36.919431,zVDYat_guuNnsZ9DemVKwA,Google/z/V/,True,[0.6924970149993896],"[[0.0, 183.14369201660156, 185.0641632080078, ...",1,yes,Y,Y,POINT (4109852.208 -142161.964)
3968,u_Zx9slVChF217zkkuGJnw_0,2021,8,NaN,NaN,-1.313337,36.872663,u_Zx9slVChF217zkkuGJnw,Google/u/_/,True,[0.40619784593582153],"[[126.26983642578125, 238.47946166992188, 328....",1,yes,Y,N,POINT (4104646.085 -146212.865)


## 1) Load raster template (flood) and built‑weight; align & sanitize

In [4]:
with rio.open(FLOOD_RASTER) as src:
    flood_profile = src.profile.copy()
    flood_crs = src.crs
    flood_transform = src.transform
    flood_shape = (src.height, src.width)
    flood_res = (src.res[0], src.res[1])

print("Flood CRS:", flood_crs)
print("Flood shape (rows, cols):", flood_shape)
print("Flood pixel size (x, y):", flood_res)

with rio.open(BUILT_WEIGHT) as bw:
    built_raw = bw.read(1)
    built_meta = bw.profile

# Sanitize built weights
built_clean = np.nan_to_num(built_raw, nan=0.0, posinf=1.0, neginf=0.0).astype("float32")
built_clean = np.clip(built_clean, 0.0, 1.0)

# Align to the flood grid
needs_resample = (
    (built_meta.get("crs") != flood_crs) or
    (built_meta.get("transform") != flood_transform) or
    (built_clean.shape != flood_shape)
)

if needs_resample:
    built = np.zeros(flood_shape, dtype="float32")
    with rio.open(BUILT_WEIGHT) as bw:
        reproject(
            source=built_clean,
            destination=built,
            src_transform=bw.transform,
            src_crs=bw.crs,
            dst_transform=flood_transform,
            dst_crs=flood_crs,
            resampling=Resampling.bilinear,
        )
    built = np.clip(np.nan_to_num(built, nan=0.0, posinf=1.0, neginf=0.0), 0.0, 1.0).astype("float32")
else:
    built = built_clean

print("Built aligned:", built.shape, "NaNs:", np.isnan(built).sum())

Flood CRS: EPSG:3857
Flood shape (rows, cols): (1416, 2100)
Flood pixel size (x, y): (38.21851414, 38.21851414)
Built aligned: (1416, 2100) NaNs: 0


## 2) Load 2 km displacement polygons and clip to raster extent

In [10]:
gdf_2km = gpd.read_file(POP_GRID_2KM)
gdf_2km = gdf_2km.to_crs(flood_crs)
missing = [f for f in POP_FIELDS if f not in gdf_2km.columns]
assert not missing, f"Missing fields in 2 km layer: {missing}"
print("2 km cells (raw):", len(gdf_2km))

AssertionError: Missing fields in 2 km layer: ['density_diff_043016_050716', 'n_diff_043016_050716', 'density_diff_050100_050800', 'n_diff_050100_050800', 'density_diff_050208_050908', 'n_diff_050208_050908', 'variation_density_sum', 'variation_density_abs_sum', 'variation_count_sum', 'variation_count_abs_sum']

In [6]:

# Clip to raster extent
height, width = flood_shape
left, top = flood_transform * (0, 0)
right, bottom = flood_transform * (width, height)
raster_extent_poly = box(min(left, right), min(top, bottom), max(left, right), max(top, bottom))
gdf_2km = gdf_2km[gdf_2km.intersects(raster_extent_poly)].copy()
print("2 km cells intersecting raster:", len(gdf_2km))

NameError: name 'gdf_2km' is not defined

## 3) Build flood mask (binary)

In [7]:
with rio.open(FLOOD_RASTER) as src:
    flood_arr = src.read(1).astype("float32")

if FLOOD_IS_BINARY:
    flood_mask = flood_arr >= 1
else:
    flood_mask = flood_arr >= FLOOD_DEPTH_THRESHOLD

print("Flooded pixel count:", int(flood_mask.sum()), "of", flood_mask.size)

Flooded pixel count: 309854 of 2973600


## 4) Dasymetric allocation helper

In [8]:
def allocate_to_30m(gdf_cells, value_field, built, flood_shape, flood_transform, when_wsum_zero="skip"):
    """Allocate a 2 km field to 30 m using weights in `built`."
    value_field can be positive/negative (population change).
    when_wsum_zero: 'skip' (recommended) or 'uniform'."""
    out = np.zeros(flood_shape, dtype="float32")
    skipped_sum = 0.0

    for _, row in gdf_cells.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        v = float(row[value_field])
        mask = features.geometry_mask([geom], out_shape=flood_shape, transform=flood_transform, invert=True)
        if not mask.any():
            continue

        W = built[mask]
        W_sum = np.nansum(W, dtype=np.float64)
        if W_sum > 0:
            out[mask] += (W / W_sum) * v
        else:
            skipped_sum += v
            if when_wsum_zero == "uniform":
                n = int(mask.sum())
                if n > 0:
                    out[mask] += (v / n)
    return out, skipped_sum

## 5) Allocate **all events** to 30 m and summarize exposure

In [9]:
alloc_rasters = {}
event_summaries = []

for field in POP_FIELDS:
    arr30, skipped = allocate_to_30m(
        gdf_cells=gdf_2km,
        value_field=field,
        built=built,
        flood_shape=flood_shape,
        flood_transform=flood_transform,
        when_wsum_zero="skip"
    )
    alloc_rasters[field] = arr30
    declared_total = float(gdf_2km[field].sum())
    allocated_total = float(np.nansum(arr30))

    net_change_in_flood  = float(arr30[flood_mask].sum())
    inflow_in_flood      = float(arr30[flood_mask][arr30[flood_mask] > 0].sum())
    outflow_in_flood_mag = float(np.abs(arr30[flood_mask][arr30[flood_mask] < 0].sum()))

    event_summaries.append({
        "event": field,
        "declared_total": declared_total,
        "allocated_total": allocated_total,
        "skipped_total": skipped,
        "net_change_in_flood": net_change_in_flood,
        "inflow_in_flood": inflow_in_flood,
        "outflow_in_flood_mag": outflow_in_flood_mag
    })

df_events = pd.DataFrame(event_summaries)
df_events

NameError: name 'gdf_2km' is not defined

## 6) Save 30 m rasters per event and a multiband stack

In [18]:
# Write individual rasters
for name, arr in alloc_rasters.items():
    out_path = WASTE_DIR/f"outputs/rasters_30m/{name}_30m.tif"
    prof = flood_profile.copy()
    prof.update(count=1, dtype="float32", nodata=0.0, compress="deflate")
    with rio.open(out_path, "w", **prof) as dst:
        dst.write(arr.astype("float32"), 1)
    print("Wrote", out_path)

# Multiband stack
order = list(alloc_rasters.keys())
stack = np.stack([alloc_rasters[k].astype("float32") for k in order], axis=0)
out_stack = WASTE_DIR/"outputs/rasters_30m/pop_change_events_stack_30m.tif"
prof = flood_profile.copy()
prof.update(count=len(order), dtype="float32", nodata=0.0, compress="deflate")
with rio.open(out_stack, "w", **prof) as dst:
    dst.write(stack)
    for i, name in enumerate(order, start=1):
        dst.set_band_description(i, name)
print("Wrote", out_stack)

Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/density_diff_043016_050716_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/n_diff_043016_050716_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/density_diff_050100_050800_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/n_diff_050100_050800_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/density_diff_050208_050908_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/n_diff_050208_050908_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/variation_density_sum_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/variation_density_abs_sum_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/outputs/rasters_30m/variation_count_sum_30m.tif
Wrote /Users/wenlanzhang/Downloads/PhD_UCL/Data/Wa

## 7) Build per‑polygon (2 km) stats per event and save `gdf_map`

In [19]:
def per_polygon_stats(gdf_cells, arr30, flood_mask, flood_shape, flood_transform, id_field=None):
    records = []
    for idx, row in gdf_cells.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        mask = features.geometry_mask([geom], out_shape=flood_shape, transform=flood_transform, invert=True)
        if not mask.any():
            continue
        inside = mask
        flood_inside = flood_mask & inside

        rec = {
            "cell_rowidx": idx if id_field is None else row[id_field],
            "alloc_sum": float(np.nansum(arr30[inside])),
            "net_change_in_flood": float(np.nansum(arr30[flood_inside])),
            "inflow_in_flood": float(arr30[flood_inside][arr30[flood_inside] > 0].sum()),
            "outflow_in_flood_mag": float(np.abs(arr30[flood_inside][arr30[flood_inside] < 0].sum())),
            "pct_area_flooded": (float(flood_inside.sum()) / float(inside.sum()) * 100.0) if inside.any() else 0.0
        }
        records.append(rec)
    return pd.DataFrame.from_records(records)

# Build wide table
all_event_stats = []
for field in POP_FIELDS:
    df_evt = per_polygon_stats(gdf_2km, alloc_rasters[field], flood_mask, flood_shape, flood_transform)
    df_evt = df_evt.add_prefix(f"{field}__")
    df_evt = df_evt.rename(columns={f"{field}__cell_rowidx": "cell_rowidx"})
    all_event_stats.append(df_evt)

df_stats_wide = all_event_stats[0]
for df_more in all_event_stats[1:]:
    df_stats_wide = df_stats_wide.merge(df_more, on="cell_rowidx", how="outer")

# Attach to geometry
gdf_map = gdf_2km.reset_index(drop=False).rename(columns={"index": "cell_rowidx"}).merge(
    df_stats_wide, on="cell_rowidx", how="left"
)
gdf_map.crs = gdf_2km.crs

gdf_map.to_file(WASTE_DIR/"outputs/nairobi_2km_event_stats.gpkg", layer="stats", driver="GPKG")
gdf_map.head()

,cell_rowidx,quadkey,density_crisis_043016,density_crisis_050100,density_crisis_050208,density_crisis_050716,density_crisis_050800,density_crisis_050908,n_crisis_043016,n_crisis_050100,...,variation_count_sum__alloc_sum,variation_count_sum__net_change_in_flood,variation_count_sum__inflow_in_flood,variation_count_sum__outflow_in_flood_mag,variation_count_sum__pct_area_flooded,variation_count_abs_sum__alloc_sum,variation_count_abs_sum__net_change_in_flood,variation_count_abs_sum__inflow_in_flood,variation_count_abs_sum__outflow_in_flood_mag,variation_count_abs_sum__pct_area_flooded
0,0,30011010220033,0.000286,0.000288,0.000191,0.000289,0.000254,0.000267,70.400451,126.973311,...,-0.564844,0.000000,0.0,0.000000,5.151367,0.714833,0.000000,0.000000,0.0,5.151367
1,1,30011010220210,0.000092,0.000101,0.000056,0.000098,0.000083,0.000088,22.672635,44.612368,...,-0.582563,-0.499537,0.0,0.499537,18.017578,0.886971,0.760562,0.760562,0.0,18.017578
2,2,30011010220211,0.000292,0.000307,0.000178,0.000284,0.000238,0.000251,72.094215,135.443096,...,-0.384077,-0.001222,0.0,0.001222,2.587891,0.831667,0.002646,0.002646,0.0,2.587891
3,3,30011010220212,0.000076,0.000074,0.000055,0.000073,0.000057,0.000078,18.703052,32.414133,...,-0.370828,-0.140906,0.0,0.140906,16.406250,0.812882,0.308876,0.308876,0.0,16.406250
4,4,30011010220213,0.000219,0.000196,0.000123,0.000204,0.000161,0.000202,54.028668,86.204820,...,-0.481566,-0.025927,0.0,0.025927,5.468750,0.789757,0.042520,0.042520,0.0,5.468750


## 8) Waste integration (30 m rasters + 2 km summaries) — optional

In [24]:
try:
    # waste_gdf = gpd.read_file(WASTE_POINTS).to_crs(gdf_2km.crs)
    print("Waste points:", len(waste_gdf))

    # (i) Rasterize hits
    waste_shapes = ((geom, 1) for geom in waste_gdf.geometry if geom and not geom.is_empty)
    waste_hits = features.rasterize(
        shapes=waste_shapes, out_shape=flood_shape, transform=flood_transform, fill=0, dtype="int16"
    )

    # (ii) Distance to nearest (m)
    background = (waste_hits == 0)
    px = float(abs(flood_transform.a))
    dist_px = ndimage.distance_transform_edt(background)
    waste_dist_m = dist_px * px

    # (iii) KDE-like density (relative 0..1)
    sigma_px = max(1.0, WASTE_KDE_BW_M / px)
    waste_kde = ndimage.gaussian_filter(waste_hits.astype("float32"), sigma=sigma_px)
    waste_kde = (waste_kde / waste_kde.max()).astype("float32") if waste_kde.max() > 0 else waste_kde.astype("float32")

    # Save rasters
    def write_raster(path, arr, profile_like, dtype="float32", nodata=0):
        prof = profile_like.copy()
        prof.update(count=1, dtype=dtype, nodata=nodata, compress="deflate")
        with rio.open(path, "w", **prof) as dst:
            dst.write(arr.astype(dtype), 1)

    write_raster(WASTE_DIR/"outputs/waste/waste_hits_30m.tif", waste_hits, flood_profile, dtype="int16", nodata=0)
    write_raster(WASTE_DIR/"outputs/waste/waste_dist_m_30m.tif", waste_dist_m, flood_profile, dtype="float32", nodata=0)
    write_raster(WASTE_DIR/"outputs/waste/waste_kde_30m.tif", waste_kde, flood_profile, dtype="float32", nodata=0)

    # 2 km summaries
    waste_rows = []
    for idx, row in gdf_2km.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        count_pts = int(waste_gdf[waste_gdf.within(geom)].shape[0])
        poly_mask = features.geometry_mask([geom], out_shape=flood_shape, transform=flood_transform, invert=True)
        if not poly_mask.any():
            continue
        min_dist = float(np.nanmin(waste_dist_m[poly_mask])) if poly_mask.any() else np.nan
        mean_kde = float(np.nanmean(waste_kde[poly_mask])) if poly_mask.any() else np.nan

        waste_rows.append({"cell_rowidx": idx, "waste_count_pts": count_pts,
                           "waste_min_dist_m": min_dist, "waste_kde_mean": mean_kde})

    df_waste2km = pd.DataFrame.from_records(waste_rows)
    gdf_map = gdf_map.merge(df_waste2km, on="cell_rowidx", how="left")
    gdf_map.to_file(WASTE_DIR/"outputs/nairobi_2km_event_stats_with_waste.gpkg", layer="stats", driver="GPKG")
    print("Waste integration complete and saved.")
except Exception as e:
    print("Waste step skipped or failed:", e)

Waste points: 3897
Waste integration complete and saved.


## 9) Optional: Pixel‑level sample & quick checks

In [25]:
# choose one event to analyze
evt_name = POP_FIELDS[0]
arr_evt = alloc_rasters[evt_name]

# sample flooded and non-flooded pixels
flood_idx = np.where(flood_mask)
non_idx   = np.where(~flood_mask)
N_max_flood = min(200_000, len(flood_idx[0]))
M_nonflood  = min(200_000, len(non_idx[0]))
sel_f = np.random.choice(len(flood_idx[0]), size=N_max_flood, replace=False) if N_max_flood>0 else []
sel_nf = np.random.choice(len(non_idx[0]), size=M_nonflood, replace=False) if M_nonflood>0 else []
rr = np.concatenate([flood_idx[0][sel_f], non_idx[0][sel_nf]])
cc = np.concatenate([flood_idx[1][sel_f], non_idx[1][sel_nf]])

df_pix = pd.DataFrame({
    "flooded": flood_mask[rr, cc].astype(np.uint8),
    "evt_change": arr_evt[rr, cc].astype(np.float32),
})
print(df_pix.describe())

             flooded    evt_change
count  400000.000000  3.925470e+05
mean        0.500000 -3.464252e-09
std         0.500001  1.496354e-07
min         0.000000 -3.993896e-06
25%         0.000000  0.000000e+00
50%         0.500000  0.000000e+00
75%         1.000000  0.000000e+00
max         1.000000  6.613405e-06


In [28]:
flood_profile["crs"]

CRS.from_wkt('PROJCS["WGS 84 / Pseudo-Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 +x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"],AUTHORITY["EPSG","3857"]]')